In [3]:
%pwd

'c:\\Users\\Dhanush Ramachandran\\Desktop\\Dhanush --\\Personal DS\\MLops\\end-to-end project\\project'

In [2]:
import os
os.chdir("../")


In [ ]:
# design of data ingestion
from dataclasses import dataclass
from pathlib import Path
import sys
import yaml

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path
    

In [ ]:
# config manager
from src.constants import *
from src.utils.common import *
class ConfiguationManager:
    def __init__(self,config_file_path = CONFIG_FILE_PATH,
                 params_file_path = PARAMS_FILE_PATH,
                 schema_file_path = SCHEMA_FILE_PATH ):
        self.configs = self.read_yaml(config_file_path)
        self.params = self.read_yaml(params_file_path)
        self.schema = self.read_yaml(schema_file_path)

        create_directories([self.configs.artifact_root])

    def get_data_ingestion_config(self)->DataIngestionConfig:
        config = self.configs.data_ingestion

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_url = config.source_url,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config

In [ ]:
# use data ingestion configs
from box import ConfigBox
from box.exceptions  import BoxValueError
import pandas as pd

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
    def load_data(self):
        # load from source url if source url is given
        if self.config.source_url:
            logger.log(f"Loading data from source url: {self.config.source_url}")
            try:
                df = pd.read_csv(self.config.source_url)
                logger.log(f"Data read successfully from {self.config.source_url}")
            except Exception as e:
                logger.log(f"Error reading CSV file from source url: {e}")
                raise BoxValueError(f"Error reading CSV file: {e}")
        else:
            # load from local data file
            try:
                logger.log(f"Loading data from local file: {self.config.local_data_file}")
                df = pd.read_csv(self.config.local_data_file)
            except Exception as e:
                logger.log(f"Error reading CSV file from local file: {e}")
                raise BoxValueError(f"Error reading csv file: {e}")
            
        return df
    